# Univariate Exploratory Data Analysis (EDA) — FT draft v3

**Week 5 · Monday · Part 1 · Estimated classroom time: 85–90 minutes**

**Business context:** We are exploring a housing dataset to understand the types of properties and the distribution of their sale prices. We will examine **one variable at a time** before studying relationships between variables in the next notebook.

### Learning goals
- Identify categorical and numerical variables, regardless of their stored dtype.
- Describe a categorical variable using counts, proportions and a clear chart.
- Describe a numerical variable using measures of center, spread and shape.
- Interpret histograms and boxplots; identify *potential* outliers without deleting them automatically.
- Write a short business interpretation supported by numerical and graphical evidence.

**Classroom convention:** short demonstrations alternate with 3–5 minute checks for understanding. Longer practice belongs to the afternoon lab.

## 1. Load and inspect the dataset (10 minutes)

We will reuse the same housing dataset in the **Bivariate EDA** notebook. The original course notebook provides the data and its dictionary:
- [Housing dataset](https://raw.githubusercontent.com/data-bootcamp-v4/data/main/housing_price_eda.csv)
- Consult the dataset documentation linked in the original lesson for the meaning of each variable.

We use a remote CSV here to preserve the original lesson's data source. In a team project, a relative local path is generally preferable.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline  # Display Matplotlib figures inside the notebook

sns.set_theme(style="whitegrid")

DATA_URL = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/housing_price_eda.csv"
df = pd.read_csv(DATA_URL)

df.head()

In [ ]:
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df[["MSZoning", "SalePrice", "GrLivArea", "TotRmsAbvGrd"]].dtypes

**Interpretation first:** `MSZoning` is a category; `SalePrice` and `GrLivArea` are quantitative; `TotRmsAbvGrd` is a count. A numeric dtype does **not** automatically mean that a variable should be analyzed as continuous. Likewise, an ID is not a measurement.

We briefly inspect missingness, but **data cleaning is not today's objective**. Do not drop rows or columns automatically based on an arbitrary threshold.

In [ ]:
df[["MSZoning", "SalePrice", "GrLivArea", "TotRmsAbvGrd"]].isna().sum()

### Check for understanding · 3 minutes

`MSZoning`, `YearBuilt`, `OverallQual`, `SalePrice` and `Id` may all be represented by numbers or strings. Which ones should be treated as categories, measurements, or identifiers? Explain **why**. (No code required.)

## 2. One categorical variable: `MSZoning` (15 minutes)

**Business question:** Which zoning classification is most common in the housing dataset?

- **Absolute frequency:** number of observations in a category.
- **Relative frequency:** fraction of observations in a category.

Use `value_counts()` for one variable. We will introduce `pd.crosstab()` tomorrow when comparing **two categorical variables**.

In [ ]:
zoning_counts = df["MSZoning"].value_counts(dropna=False)
zoning_proportions = df["MSZoning"].value_counts(normalize=True, dropna=False)

zoning_summary = pd.DataFrame({
    "count": zoning_counts,
    "proportion": zoning_proportions,
    "percentage": zoning_proportions * 100,
})
zoning_summary

### A single, readable chart

For categorical data, a bar chart is usually easier to compare than a pie chart. We use **Seaborn** for the chart and Matplotlib only when additional layout or export control is useful.

In [ ]:
ax = sns.countplot(data=df, y="MSZoning", order=df["MSZoning"].value_counts().index)
ax.set(title="Homes by zoning classification", xlabel="Number of homes", ylabel="Zoning")
plt.show()

### Check for understanding · 4 minutes

1. Which zoning classification is most common? Approximately what percentage does it represent?
2. What would change if we plotted **proportions** rather than **counts**?
3. Would a pie chart make the comparison easier or harder? Explain your choice.

## 3. One numerical variable: `SalePrice` (20 minutes)

**Business question:** What is a typical sale price, and how much do prices vary?

Start with `describe()`, then calculate selected statistics explicitly. The **mean** and **median** answer related but different questions. A skewed distribution can make the mean less representative of a typical observation.

In [ ]:
prices = df["SalePrice"].dropna()
prices.describe()

In [ ]:
mean_price = prices.mean()
median_price = prices.median()
mode_price = prices.mode().iloc[0]  # The first mode, if several exist

print(f"Mean:   ${mean_price:,.0f}")
print(f"Median: ${median_price:,.0f}")
print(f"Mode:   ${mode_price:,.0f}")

### Spread and quantiles

- **Range** = maximum − minimum; sensitive to extremes.
- **Variance** = average squared deviation (with sample correction in Pandas); expressed in squared units.
- **Standard deviation** = square root of variance; expressed in the original units.
- **IQR** = Q3 − Q1; describes the middle 50% of observations and is less sensitive to extreme values.

In [ ]:
q1, q3 = prices.quantile([0.25, 0.75])
iqr = q3 - q1

spread = pd.Series({
    "range": prices.max() - prices.min(),
    "sample_variance": prices.var(ddof=1),
    "sample_std": prices.std(ddof=1),
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
})
spread

### Check for understanding · 4 minutes

Suppose a few very expensive homes increase the **mean** noticeably but barely change the **median**. Which measure would you report as the price of a *typical* home? Why?

What does the IQR tell you that the full range does not?

## 4. Shape: histogram, skewness and kurtosis (15 minutes)

A histogram groups numerical observations into intervals (**bins**) to reveal the overall shape of their distribution. Too few bins may hide structure; too many may emphasize random fluctuations.

There is no universally optimal number of bins. We use **`bins="auto"`**: NumPy compares the Sturges and Freedman–Diaconis rules and selects the **smaller bin width** (the more detailed partition). This is a useful starting point, not a guarantee of the best histogram.

As sample size increases, a histogram can provide a more reliable picture of the underlying distribution **provided the bin width is chosen appropriately**. The histogram is still an estimate: its shape and any inferred tail probabilities depend on the binning and the available observations.

With **`kde=True`**, Seaborn overlays a *kernel density estimate* (KDE): a smooth estimate of the probability density. The curve is not the true distribution or a normality test; its appearance depends on the smoothing bandwidth.

**Skewness** describes asymmetry. **Excess kurtosis** (as returned by Pandas) is related to tail weight and extreme observations relative to a normal distribution; it is **not** simply a measure of how tall the peak is.


In [ ]:
print(f"Skewness: {prices.skew():.2f}")
print(f"Excess kurtosis: {prices.kurtosis():.2f}")

ax = sns.histplot(prices, bins="auto", kde=True)
ax.set(title="Distribution of sale prices", xlabel="Sale price ($)", ylabel="Number of homes")
plt.show()

### Check for understanding · 3 minutes

Does the histogram suggest positive or negative skewness? Is that consistent with the mean–median comparison? Identify **one visible feature** supporting your answer.

## 5. Boxplots and potential outliers (15 minutes)

A boxplot displays the median, Q1, Q3 and whiskers. Under the usual **Tukey convention**, observations beyond **Q1 − 1.5 × IQR** or **Q3 + 1.5 × IQR** are flagged as *potential* outliers.

**Important:** The Tukey rule does **not directly measure the probabilistic rarity of an observation**. In skewed or heavy-tailed distributions, valid observations may lie beyond the whiskers. The same caution applies to rules based on a fixed number of standard deviations.

Probabilistic rarity would involve the probability of observing values **as extreme or more extreme** under an appropriate reference distribution. There is no universal probability threshold (such as 5% or 1%) for calling something an outlier: context, distributional assumptions and sample size matter.

**Detecting, investigating and removing an outlier are three different decisions.** A flagged value deserves inspection; it should not be removed automatically. For example, an unusually expensive house may be a valid sale rather than a data-entry error.


In [ ]:
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
flagged = prices[(prices < lower_fence) | (prices > upper_fence)]

print(f"Lower fence: ${lower_fence:,.0f}")
print(f"Upper fence: ${upper_fence:,.0f}")
print(f"Potential outliers: {len(flagged):,} of {len(prices):,} observations")

ax = sns.boxplot(x=prices)
ax.set(title="Sale prices: center, spread and potential outliers", xlabel="Sale price ($)")
plt.show()

### Comparing figures fairly

When plotting the **same variable** in multiple panels, use shared axes where appropriate (`sharex=True`, `sharey=True`) to avoid misleading differences in scale. Do **not** force shared axes on unrelated variables with different units.

`plt.subplots()` returns a single `Axes` object for one subplot. With several subplots, `axes` is normally a NumPy array of `Axes` objects. We therefore use `axes[0]` and `axes[1]` to specify where each Seaborn chart should be drawn using the `ax=` argument.

`plt.tight_layout()` helps prevent overlap between subplot titles, labels and axes. Save figures **before** `plt.show()` and use relative paths. The next cell illustrates this without requiring any absolute path.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

# `axes` is a NumPy array: axes[0] is the top panel, axes[1] the bottom.
sns.histplot(prices, bins="auto", ax=axes[0])
sns.boxplot(x=prices, ax=axes[1])
axes[0].set(title="Sale price distribution", xlabel="", ylabel="Count")
axes[1].set(xlabel="Sale price ($)")

plt.tight_layout()

# Relative to the notebook's current working directory:
output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)
plt.savefig(output_dir / "sale_price_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

### Check for understanding · 4 minutes

A house lies above the upper Tukey fence. Should we remove it from the dataset? What would you check before deciding?

## 6. Wrap-up: turn analysis into a business insight (5 minutes)

A useful EDA conclusion should include:
1. **Observation** — what the data or plot shows.
2. **Evidence** — a statistic, count, percentage or chart.
3. **Interpretation** — why it might matter for the business.
4. **Caution** — what the analysis does *not* establish.

### Final check for understanding · 4 minutes

Write **two or three sentences** describing the distribution of `SalePrice`. Mention the mean/median relationship, spread or skewness, and one limitation of the analysis.

**Next:** In *Bivariate EDA*, we will compare two variables at a time: categorical–categorical, categorical–numerical and numerical–numerical.

---
## Optional reference — not required in class

### Discretization with `pd.cut()` and `pd.qcut()`

These methods turn a numerical variable into categories. They can be useful for reporting, but grouping discards information. `cut()` uses defined boundaries; `qcut()` uses quantiles to obtain approximately balanced groups (subject to ties).

```python
# Example only: not needed for today's lab
price_bands = pd.cut(prices, bins=[0, 100_000, 200_000, 300_000, float("inf")])
price_quartiles = pd.qcut(prices, q=4, duplicates="drop")
```

### Pie charts

Pie charts are available, but bar charts are generally preferable for precise category comparisons. The univariate lab includes a pie-chart exercise, which students can solve using `Series.plot.pie()` or Matplotlib documentation.
